# 01 - Intro to DMD with toy data

Before using bird keypoints, this notebook builds a small dataset where we already know the answer.

The toy animal is deliberately simple:

- the **left arm** waves at **2 Hz** with constant amplitude. Its DMD eigenvalues should sit on the unit circle.
- the **right arm** waves at **5 Hz** but fades out through time. Its DMD eigenvalues should sit inside the unit circle.
- a smaller **shared coordination** wave moves both arms and the body at **3 Hz**. It is deliberately less obvious in a single marker trace, but DMD should recover it as one coordinated spatial mode pair.

We will run the same workflow used later on the hawk data: PCA/SVD, FFT, then DMD.

## Setup

Run the next cell once at the start. It installs the workshop package and downloads the small workshop files. After that, the notebook uses ordinary Python for the DMD story.

In [ ]:
# Run this once at the start of Colab.
!pip install -q uv
!uv pip install --system -q git+https://github.com/LydiaFrance/dmd-workshop.git@main

from animal_dmd import setup_workshop
setup_workshop();

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from birddmd import run_dmd

from animal_dmd import (
    make_toy_motion,
    plot_combined_toy_motion,
    plot_marker_traces,
    plot_reconstructed_pair_motions,
    plot_reconstruction_marker_traces,
    plot_short_clip_fft_limit,
    plot_svd_summary,
    plot_toy_components,
    plot_toy_svd_component_shapes,
    plot_trace_fft_comparison,
    plot_unit_circle_eigenvalues,
    print_dmd_summary,
    reconstruct_known_pairs,
)

np.set_printoptions(precision=3, suppress=True)

# This removes a warning about the data condition number (referring to the spread)
warnings.filterwarnings(
    "ignore",
    message="Input data condition number inf.*",
    category=UserWarning,
)

## Build a motion dataset where we know the answer

Each row in `mean_shape` is one tracked point. The toy data is shaped like real keypoint data: `(frames, markers, xyz)`.

DMD sees each frame as one long **state vector**. For example, 7 markers with 3 coordinates become 21 numbers per frame. That flattening step is only bookkeeping: one time point becomes one column of numbers.

This example is intentionally artificial. We are not trying to model a real animal yet. We are making three clean motions so the DMD terminology has something concrete to attach to: one steady oscillator, one decaying oscillator, and one smaller coordinated oscillator that affects several markers at once.

You set the three frequencies in the next cell. Pick any three, roughly 1-8 Hz and a couple of Hz apart so they stay easy to tell apart; DMD will try to recover them later from the motion alone.

In [ ]:
# Frequencies (Hz) hidden in the motion. Pick any three, roughly 1-8 Hz and a
# couple of Hz apart.
steady_hz = 2.0   # left arm, steady
shared_hz = 3.0   # shared across the body and both arms
decay_hz  = 5.0   # right arm, fast but fading

dt = 0.01  # seconds between frames
times = np.arange(0, 4, dt)

decay_per_second = -0.8  # how fast the right arm fades (0 = no decay)

# Motion 1: the left arm waves steadily.
steady_wave = np.cos(2 * np.pi * steady_hz * times)
steady_sideways = np.sin(2 * np.pi * steady_hz * times)

# Motion 2: a smaller coordinated motion shared across both arms and the body.
shared_wave = np.cos(2 * np.pi * shared_hz * times)
shared_sideways = np.sin(2 * np.pi * shared_hz * times)

# Motion 3: the right arm waves faster, but decays.
decay_envelope = np.exp(decay_per_second * times)
decaying_wave = decay_envelope * np.cos(2 * np.pi * decay_hz * times)
decaying_sideways = decay_envelope * np.sin(2 * np.pi * decay_hz * times)

# A non-motion: zero amplitude for all frames.
zero_wave = np.zeros_like(times)

# Generate the toy motion from all the time-varying waves.
motion, mean_shape, marker_names = make_toy_motion(
    steady_wave,
    steady_sideways,
    shared_wave,
    shared_sideways,
    decaying_wave,
    decaying_sideways,
)

print("motion shape:", motion.shape)
print("state vector length:", motion.shape[1] * motion.shape[2])

### The Toy Motion

First look at the combined movement: all three simple motions are happening at once.

In [ ]:
plot_combined_toy_motion(
    mean_shape,
    motion,
    dt=dt,
)
plt.show()

### Splitting out the Different Motions

Because we built this motion from three parts, we can have a look at the parts separately. We can only do this because it is a toy example!

In [ ]:
# ----
# Because we defined the movement we can also 
# split out the motion for different parts in order to look at it

# Same stick figure but only the steady motion of the left arm
left_arm_only, _, _ = make_toy_motion(
    steady_wave,
    steady_sideways,
    zero_wave,
    zero_wave,
    zero_wave,
    zero_wave,
)
# Same stick figure but only the small-amplitude shared motion
shared_motion_only, _, _ = make_toy_motion(
    zero_wave,
    zero_wave,
    shared_wave,
    shared_sideways,
    zero_wave,
    zero_wave,
)
# Same stick figure but only the decaying motion of the right arm
right_arm_only, _, _ = make_toy_motion(
    zero_wave,
    zero_wave,
    zero_wave,
    zero_wave,
    decaying_wave,
    decaying_sideways,
)

## A three panel plot with each motion

plot_toy_components(
    mean_shape,
    left_arm_only,
    shared_motion_only,
    right_arm_only,
    dt=dt,
    steady_hz=steady_hz,
    shared_hz=shared_hz,
    decay_hz=decay_hz,
)
plt.show()

In [ ]:
body_marker_index = list(marker_names).index("body")
left_hand_index = list(marker_names).index("left_hand")
right_hand_index = list(marker_names).index("right_hand")

plot_marker_traces(
    times,
    {
        "left hand y: steady 2 Hz plus shared 3 Hz": motion[:, left_hand_index, 1],
        "right hand y: decaying 5 Hz plus shared 3 Hz": motion[:, right_hand_index, 1],
        "body y: mostly shared 3 Hz": motion[:, body_marker_index, 1],
    },
    title="Three marker traces from the toy motion",
)
plt.show()

## Make the analysis matrix

This is the main shape change in the notebook.

- `motion` has shape `(frames, markers, xyz)`, which is convenient for plotting.
- `data_matrix` has shape `(coordinates, frames)`, which is convenient for PCA and DMD.

The mean is removed because we want to analyse motion around the average pose, not the average pose itself.

In [ ]:
mean_pose = motion.mean(axis=0, keepdims=True)
centred_motion = motion - mean_pose
data_matrix = centred_motion.reshape(centred_motion.shape[0], -1).T

print("motion:", motion.shape)
print("centred motion:", centred_motion.shape)
print("data matrix:", data_matrix.shape)

## PCA/SVD: good spatial compression, no built-in frequency labels

SVD finds directions that explain variance. That is useful, but a component can mix more than one frequency when the data is not perfectly separated by variance.

In this toy, the small shared 3 Hz motion is useful: because it is spread across several markers, it is not necessarily the largest thing in any one trace, but it is still a real coordinated motion.

In [ ]:
spatial_patterns, singular_values, temporal_scores = np.linalg.svd(data_matrix, full_matrices=False)
variance_explained = singular_values**2 / np.sum(singular_values**2)

plot_svd_summary(
    times,
    variance_explained,
    temporal_scores,
    n_bars=8,
    title="How compact is the motion?",
)
plt.show()

print("First four variance percentages:", np.round(variance_explained[:4] * 100, 1))

SVD also gives spatial directions. To make that concrete, we can reshape each component back into marker coordinates and move the mean shape in the negative and positive component directions.

The sign is arbitrary: “negative” and “positive” are just the two ends of the same component.

In [ ]:
plot_toy_svd_component_shapes(
    mean_shape,
    spatial_patterns,
    singular_values,
    temporal_scores,
    n_components=3,
)
plt.show()

We built three coordinated patterns into it: the left hand has its own 2 Hz x-y loop, the body/head/tail and both hands share a smaller 3 Hz in-phase motion, and the right hand has its own 5 Hz x-y loop that decays through time.

The SVD spatial components are therefore correctly showing marker coordinates moving together. The leading component mostly describes the large left-hand loop; next component picks up the right-hand loop; and the smaller shared 3 Hz cross-body motion is present but spread across components rather than isolated as one perfectly labelled source. That is expected: PCA/SVD organises motion by variance and orthogonal spatial directions. It is good for asking **which parts of the body move together**, while DMD asks **what dynamic component explains that coordinated motion**.


## FFT: good frequency labels for one signal, no full spatial mode

FFT asks: which frequencies are present in this one trace? That is powerful, but the answer depends on which marker and coordinate you choose.

The plot below uses one small FFT for each selected trace, with each trace normalised to its own maximum. That makes the limitation visible: the left hand mostly says "2 Hz", the body mostly says "3 Hz", and the right hand mostly says "5 Hz". FFT can label frequencies in traces, but it does not by itself say which markers belong to the same coordinated motion.

That is the gap DMD is meant to fill: it connects a frequency to a coordinated shape change.

In [ ]:

# Running FFT on different markers. 
toy_marker_names = list(marker_names)
left_hand_y =  motion[:, toy_marker_names.index("left_hand"), 1]
right_hand_y = motion[:, toy_marker_names.index("right_hand"), 1]
body_y =       motion[:, toy_marker_names.index("body"), 1]

n_frames = motion.shape[0]
fft_frequencies_hz = np.fft.rfftfreq(n_frames, d=dt)

left_hand_fft =  np.fft.rfft(left_hand_y - left_hand_y.mean())
body_fft =       np.fft.rfft(body_y - body_y.mean())
right_hand_fft = np.fft.rfft(right_hand_y - right_hand_y.mean())

# The power of the FFT is the square of the amplitude.
trace_power = {
    "left hand y": np.abs(left_hand_fft) ** 2,
    "shared/body y": np.abs(body_fft) ** 2,
    "right hand y": np.abs(right_hand_fft) ** 2,
}
# What we built into the toy.
true_frequencies = {
    "steady 2 Hz": (steady_hz, "tab:blue"),
    "shared 3 Hz": (shared_hz, "tab:green"),
    "decaying 5 Hz": (decay_hz, "tab:orange"),
}

plot_trace_fft_comparison(fft_frequencies_hz, trace_power, true_frequencies=true_frequencies)
plt.show()


### FFT also needs enough repeated cycles

There is another practical limitation. FFT works best when the clip is long enough and the motion is roughly steady across the whole clip.

A short clip has coarse frequency bins. The spacing between FFT bins is about `1 / clip_duration`. In a 4 second clip the bins are 0.25 Hz apart, so 2 Hz and 3 Hz can be separated. In a 0.75 second clip the bins are about 1.33 Hz apart, so the same signal has much less room to put nearby 2 Hz and 3 Hz components cleanly.

This matters for animal data because we often have short manoeuvres, partial wingbeats, starts and stops, or movements that fade in and out. The decaying 5 Hz trace above is another example: changing amplitude spreads power around the main frequency instead of making one perfect spike. FFT can still be useful, but it is not automatically a clean description of the movement. DMD also needs enough frames, but when the data are sufficient it fits frequency, growth/decay, and spatial pattern together.


In [ ]:
short_duration_seconds = 0.75
short_frame_count = int(short_duration_seconds / dt)
short_left_hand_y = left_hand_y[:short_frame_count]

full_clip_frequencies_hz = np.fft.rfftfreq(left_hand_y.size, d=dt)
full_clip_power = np.abs(np.fft.rfft(left_hand_y - left_hand_y.mean())) ** 2

short_clip_frequencies_hz = np.fft.rfftfreq(short_left_hand_y.size, d=dt)
short_clip_power = np.abs(np.fft.rfft(short_left_hand_y - short_left_hand_y.mean())) ** 2

flight_biology_frequencies = {
    "steady 2 Hz": (steady_hz, "tab:blue"),
    "shared 3 Hz": (shared_hz, "tab:green"),
}

plot_short_clip_fft_limit(
    (full_clip_frequencies_hz, full_clip_power),
    (short_clip_frequencies_hz, short_clip_power),
    full_duration_seconds=times.size * dt,
    short_duration_seconds=short_duration_seconds,
    marked_frequencies=flight_biology_frequencies,
)
plt.show()

## DMD: spatial modes plus temporal dynamics

Here is the non-equation version first.

DMD tries to describe the full motion as a small set of repeatable motions. Each DMD motion has two parts:

- **where things move**: the mode, a pattern across all markers and coordinates
- **how that pattern changes in time**: its frequency and whether it grows, fades, or stays steady

The first DMD idea is simple: find a rule that advances the motion by one frame. If `x_k` is the flattened marker state at frame `k`, then DMD fits:

$$
 x_{k+1} \approx A x_k
$$

Read this as: "one matrix `A` approximately steps the whole posture forward by one frame".

The special patterns of that matrix are the DMD **modes**:

$$
 A \phi_j = \mu_j \phi_j
$$

Read this as: "mode $\phi_j$ keeps its shape when the system advances; it only gets multiplied by $\mu_j$".

For interpretation, it is usually easier to write the reconstruction over continuous time:

$$
 x(t) \approx \sum_j \phi_j\,\psi_j\,e^{\lambda_j t}
$$

Read this as: "rebuild the motion by adding up modes; each mode has a starting size and a clock".

The symbols mean:

- $\phi_j$ (`phi`) is the **mode**: the coordinated spatial pattern across all markers and coordinates
- $\psi_j$ (`psi`) is the **amplitude**: how much that mode contributes at the start
- $\lambda_j$ (`lambda`) is the continuous-time eigenvalue: the mode's clock
- $\lambda_j = \sigma_j + i\omega_j$: $\sigma$ is growth/decay, and $\omega$ (`omega`) is angular frequency in radians per second
- frequency in Hz is $f_j = \omega_j / (2\pi)$
- the per-frame eigenvalue is $\mu_j = e^{\lambda_j\,dt}$

The advantage over one FFT trace is that DMD gives a **frequency plus the whole spatial pattern moving at that frequency**. That is what we need for animal keypoints: wingbeat is not one number, it is a coordinated motion of many markers.

After fitting, the first sanity check is the full reconstruction: if we add the DMD modes back together, do we recover the marker traces we started from?

For discrete-time DMD, the unit circle is the useful visual guide:

- on the circle: constant amplitude oscillation
- inside the circle: decays
- outside the circle: grows

We start with `dmd_rank = 6`: three oscillations make three conjugate pairs, and each pair needs two modes. Try 4 and 10 as well, and compare the recovered frequencies and the RMSE.

In [ ]:
dmd_rank = 6  # 3 hidden motions -> 3 pairs -> rank 6; try 4 and 10
dmd_result = run_dmd(
    data=motion,
    times=times,
    n_modes=dmd_rank,
    d=2,
    eig_constraints={"conjugate_pairs"},
    average_shape=mean_pose,
    n_markers=motion.shape[1],
    verbose=False,
)
reconstructed_motion = dmd_result.reconstruction
reconstruction_rmse = np.sqrt(np.mean((motion[1:] - reconstructed_motion) ** 2))

growth_per_second = dmd_result.eigenvalues.real
omega_radians_per_second = dmd_result.eigenvalues.imag
frequency_hz = omega_radians_per_second / (2 * np.pi)

print_dmd_summary(growth_per_second, frequency_hz, reconstruction_rmse)

print("Set frequencies:", sorted([steady_hz, shared_hz, decay_hz]), "Hz")
print("DMD recovered:  ", sorted(round(float(f), 2) for f in np.abs(frequency_hz[frequency_hz > 0])), "Hz")

reconstruction_times = times[1:]
measured_reconstruction_traces = {
    "left hand y": motion[1:, left_hand_index, 1],
    "shared/body y": motion[1:, body_marker_index, 1],
    "right hand y": motion[1:, right_hand_index, 1],
}
dmd_reconstruction_traces = {
    "left hand y": reconstructed_motion[:, left_hand_index, 1],
    "shared/body y": reconstructed_motion[:, body_marker_index, 1],
    "right hand y": reconstructed_motion[:, right_hand_index, 1],
}

plot_reconstruction_marker_traces(
    reconstruction_times,
    measured_reconstruction_traces,
    dmd_reconstruction_traces,
    title="Full DMD reconstruction of marker traces",
)
plt.show()

In [ ]:
per_frame_eigenvalues = np.exp(dmd_result.eigenvalues * dt)
plot_unit_circle_eigenvalues(per_frame_eigenvalues)
plt.show()

BirdDMD reports continuous-time eigenvalues, $\lambda$. For the unit-circle plot, we convert them to per-frame eigenvalues with:

$$
 \mu = e^{\lambda\,dt}
$$

Plainly: $\lambda$ describes what happens per second, while $\mu$ describes what happens per frame.

The 2 Hz pair and the shared 3 Hz pair should be almost exactly on the circle because they have constant amplitude. The 5 Hz pair should be inside the circle because we made it decay.

The radius panel on the right is deliberately not a normal complex-plane plot. It zooms into $|\mu|$, the radius from the origin, because the decaying mode is only slightly inside the unit circle over one 0.01 s frame.

## Reconstruct one DMD pair at a time

A real back-and-forth oscillation usually appears as a positive-frequency and negative-frequency pair. This is not two separate animal movements. It is how a real wave is represented with complex numbers: one member rotates one way in the complex plane, the other rotates the opposite way, and together they add up to one real movement we can draw.

That is why we inspect **pairs** rather than isolated DMD modes for oscillations. Keeping only one member of the pair would leave a complex half-description; keeping both gives the real spatial motion.

To inspect one movement, we keep both members of one pair and set the other amplitudes to zero.

This is where $\phi$, $\psi$, $\lambda$, and $\omega$ become practical:

- the selected pair's $\omega$ gives the oscillation frequency
- the selected pair's $\phi$ gives the marker pattern
- the selected pair's $\psi$ gives its starting contribution
- the selected pair's $\lambda$ controls both frequency and growth/decay through time

If DMD has separated the toy correctly, the reconstructed pairs should look like the three motions we built: left-arm 2 Hz, shared 3 Hz, and decaying right-arm 5 Hz. The plot below shows each pair in two ways: the top row is the spatial pattern, and the lower row is a representative waveform so the frequency and decay are visible.

In [ ]:
known_dmd_pairs_hz = {
    "2 Hz steady left-arm pair": steady_hz,
    "3 Hz shared coordination pair": shared_hz,
    "5 Hz decaying right-arm pair": decay_hz,
}

pair_frequencies_hz, dmd_pairs, selected_pair_frequencies_hz, reconstructed_pair_motions = reconstruct_known_pairs(
    dmd_result,
    known_dmd_pairs_hz,
)

print(dmd_pairs)
print("pair frequencies:", np.round(pair_frequencies_hz, 3))


In [ ]:
plot_reconstructed_pair_motions(
    mean_shape,
    reconstructed_pair_motions,
    times=times,
    built_in_waveforms={
        "left hand y": left_arm_only[:, left_hand_index, 1],
        "shared/body y": shared_motion_only[:, body_marker_index, 1],
        "right hand y": right_arm_only[:, right_hand_index, 1],
    },
    trace_marker_indices=[left_hand_index, body_marker_index, right_hand_index],
    phase_frequencies_hz=[steady_hz, shared_hz, decay_hz],
    dmd_frequencies_hz=selected_pair_frequencies_hz,
    dt=dt,
    colours=["tab:blue", "tab:green", "tab:orange"],
)
plt.show()


### A shorter clip: DMD still needs enough data

Now return to the same 0.75 second window used in the short FFT example. This is still a clean toy problem, not a general promise that DMD can solve every short recording. The point is narrower: with enough spatial structure and enough frames, DMD can fit the time behaviour and marker pattern together even when the short FFT view is hard to read.


In [ ]:
short_motion = motion[:short_frame_count]
short_times = times[:short_frame_count]
short_mean_pose = short_motion.mean(axis=0, keepdims=True)

short_dmd_result = run_dmd(
    data=short_motion,
    times=short_times,
    n_modes=dmd_rank,
    d=2,
    eig_constraints={"conjugate_pairs"},
    average_shape=short_mean_pose,
    n_markers=short_motion.shape[1],
    verbose=False,
)
short_reconstruction_rmse = np.sqrt(np.mean((short_motion[1:] - short_dmd_result.reconstruction) ** 2))

short_pair_frequencies_hz, short_dmd_pairs, short_selected_pair_frequencies_hz, short_reconstructed_pair_motions = reconstruct_known_pairs(
    short_dmd_result,
    known_dmd_pairs_hz,
)

print("short-window pair frequencies:", np.round(short_pair_frequencies_hz, 3))
print("short-window reconstruction RMSE:", round(short_reconstruction_rmse, 5))


In [ ]:
plot_reconstructed_pair_motions(
    mean_shape,
    short_reconstructed_pair_motions,
    times=short_times,
    built_in_waveforms={
        "left hand y": left_arm_only[:short_frame_count, left_hand_index, 1],
        "shared/body y": shared_motion_only[:short_frame_count, body_marker_index, 1],
        "right hand y": right_arm_only[:short_frame_count, right_hand_index, 1],
    },
    trace_marker_indices=[left_hand_index, body_marker_index, right_hand_index],
    phase_frequencies_hz=[steady_hz, shared_hz, decay_hz],
    dmd_frequencies_hz=short_selected_pair_frequencies_hz,
    dt=dt,
    colours=["tab:blue", "tab:green", "tab:orange"],
)
plt.show()


## What we have learnt before we go to animal data

The hawk data is more complicated, but the vocabulary is the same.

- **state vector** $x(t)$: one flattened frame of marker coordinates
- **mode** $\phi$: a coordinated spatial pattern across markers
- **amplitude** $\psi$: how strongly that mode contributes at the start
- **eigenvalue** $\lambda = \sigma + i\omega$: the time behaviour attached to the mode
- **growth/decay** $\sigma$: the real part of $\lambda$
- **frequency** $f = \omega/(2\pi)$: the imaginary part of $\lambda$ converted from radians per second to cycles per second
- **per-frame eigenvalue** $\mu = e^{\lambda dt}$: what we plot against the unit circle
- **conjugate pair**: the positive- and negative-frequency DMD modes whose complex parts cancel, producing one observable oscillatory motion in the animal coordinates

The useful promise of DMD is not just "it finds frequencies". FFT already does that for a chosen trace. The useful promise is that DMD links each frequency to a full-body spatial pattern, so we can reconstruct, remove, recombine, or extend coordinated motions in the next notebook.